<a href="https://colab.research.google.com/github/ameliagarciarond22-source/Modelos-Estoc-sticos-/blob/main/M%C3%A9todo_de_Uniformizaci%C3%B3n_para_una_CMTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Método de Uniformización
## Objetivo

Implementar el método de uniformización para aproximar la matriz de transición

$$
P(t)=\sum_{k=0}^{\infty}
e^{-rt}\frac{(rt)^k}{k!}\hat P^k
$$

y verificar la ecuación de Chapman-Kolmogorov.

---

In [1]:
import numpy as np
from scipy.stats import poisson
from math import exp, factorial

In [3]:
## Definición de la matriz de tasas

R = np.array([
    [0,2,3,0],
    [4,0,2,0],
    [0,2,0,2],
    [1,0,3,0]
], dtype=float)

print("Matriz R:")
print(R)

Matriz R:
[[0. 2. 3. 0.]
 [4. 0. 2. 0.]
 [0. 2. 0. 2.]
 [1. 0. 3. 0.]]


In [4]:
## Cálculo de r y de la matriz uniformizada P̂

ri = np.sum(R, axis=1)

r = np.max(ri)

n = len(R)

P_hat = np.zeros((n,n))

for i in range(n):
    for j in range(n):

        if i == j:
            P_hat[i,j] = 1 - ri[i]/r

        else:
            P_hat[i,j] = R[i,j]/r

print("r =", r)

print("\nP̂ =")
print(P_hat)

r = 6.0

P̂ =
[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]


# Ejercicio 3.1

Implementación de la serie de uniformización

$$
P(t)=
\sum_{k=0}^{M}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^k
$$

utilizando

$$
M \approx \max\{rt+5\sqrt{rt},20\}
$$

In [5]:
def uniformizacion(R, t):

    ri = np.sum(R, axis=1)
    r = np.max(ri)

    n = len(R)

    P_hat = np.zeros((n,n))

    for i in range(n):
        for j in range(n):

            if i == j:
                P_hat[i,j] = 1 - ri[i]/r
            else:
                P_hat[i,j] = R[i,j]/r

    M = int(max(r*t + 5*np.sqrt(r*t),20))

    P = np.zeros((n,n))

    for k in range(M+1):

        coef = exp(-r*t)*(r*t)**k/factorial(k)

        P += coef*np.linalg.matrix_power(P_hat,k)

    return P, M

In [6]:
for t in [0.5,1,5]:

    P,M = uniformizacion(R,t)

    print("="*60)
    print(f"t = {t}")
    print(f"M = {M}")
    print(P)
    print()

t = 0.5
M = 20
[[0.25060868 0.2169646  0.38665694 0.14576979]
 [0.25313484 0.23836098 0.37440924 0.13409493]
 [0.1691195  0.19361489 0.42030102 0.2169646 ]
 [0.15801748 0.15744464 0.39833179 0.28620609]]

t = 1
M = 20
[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]

t = 5
M = 57
[[0.19999925 0.19999925 0.3999985  0.19999925]
 [0.19999925 0.19999925 0.3999985  0.19999925]
 [0.19999925 0.19999925 0.3999985  0.19999925]
 [0.19999925 0.19999925 0.3999985  0.19999925]]



# Ejercicio 3.2

Verificación de Chapman-Kolmogorov

$$
P(1)=P(0.5)P(0.5)
$$

In [7]:
P05,_ = uniformizacion(R,0.5)

P1,_ = uniformizacion(R,1)

producto = P05 @ P05

print("P(1)")
print(P1)

print("\nP(0.5)P(0.5)")
print(producto)

print("\nError absoluto máximo:")

error = np.max(np.abs(P1-producto))

print(error)

P(1)
[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]

P(0.5)P(0.5)
[[0.20615141 0.20390232 0.39871018 0.19123609]
 [0.20828451 0.20534099 0.39789976 0.18847475]
 [0.19675878 0.19837963 0.40095927 0.20390232]
 [0.19204651 0.19399744 0.40147152 0.21248452]]

Error absoluto máximo:
5.820333639494635e-07


# Ejercicio 4.2

Algoritmo de uniformización con tolerancia

$$
\varepsilon = 10^{-5}
$$

In [8]:
def uniformizacion_eps(R,t,eps=1e-5):

    ri = np.sum(R,axis=1)

    r = np.max(ri)

    n = len(R)

    P_hat = np.zeros((n,n))

    for i in range(n):
        for j in range(n):

            if i == j:
                P_hat[i,j] = 1-ri[i]/r
            else:
                P_hat[i,j] = R[i,j]/r

    A = P_hat.copy()

    B = exp(-r*t)*np.eye(n)

    c = exp(-r*t)

    suma = c

    k = 1

    while suma < 1-eps:

        c = c*(r*t)/k

        B += c*A

        A = A@P_hat

        suma += c

        k += 1

    M = k-1

    return B,M

In [9]:
for t in [0.5,1,5]:

    P,M = uniformizacion_eps(R,t)

    print("="*60)
    print(f"t = {t}")
    print(f"M utilizado = {M}")
    print(P)
    print()

t = 0.5
M utilizado = 13
[[0.250608   0.21696392 0.38665557 0.14576911]
 [0.25313416 0.2383603  0.37440788 0.13409425]
 [0.16911882 0.19361421 0.42029966 0.21696392]
 [0.1580168  0.15744396 0.39833043 0.28620541]]

t = 1
M utilizado = 19
[[0.20615038 0.20390128 0.39870811 0.19123506]
 [0.20828347 0.20533995 0.39789768 0.18847371]
 [0.19675775 0.19837859 0.4009572  0.20390128]
 [0.19204548 0.1939964  0.40146945 0.21248349]]

t = 5
M utilizado = 56
[[0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999852 0.19999852 0.39999705 0.19999853]
 [0.19999852 0.19999852 0.39999705 0.19999853]]



In [10]:
# Comparación de métodos

for t in [0.5,1,5]:

    P1,M1 = uniformizacion(R,t)

    P2,M2 = uniformizacion_eps(R,t)

    error = np.max(np.abs(P1-P2))

    print("="*60)
    print(f"t = {t}")
    print(f"M serie = {M1}")
    print(f"M tolerancia = {M2}")
    print(f"Error máximo = {error}")
    print()

t = 0.5
M serie = 20
M tolerancia = 13
Error máximo = 1.3607613894017767e-06

t = 1
M serie = 20
M tolerancia = 19
Error máximo = 1.4900247798932398e-06

t = 5
M serie = 57
M tolerancia = 56
Error máximo = 1.4500849976339936e-06

